<a href="https://colab.research.google.com/github/kenleefk-edu/C3669C-2026-05/blob/main/Copy_of_Qwen3_VoiceCloning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ad089557`
import os

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
print("PYTORCH_CUDA_ALLOC_CONF set to expandable_segments:True")

` # d86fd6d6`
# Task
Install the required Python packages 'qwen-tts', 'soundfile', and 'numpy'.

` # 218fc16a`
## Install Qwen-TTS Dependencies

### Subtask:
Install the required Python packages 'qwen-tts', 'soundfile', and 'numpy' using pip. This will resolve the 'ModuleNotFoundError' encountered during the previous execution.


` # 6f63ef05`  
**Reasoning**:
The previous execution failed due to a ModuleNotFoundError for 'qwen_tts'. To resolve this, I need to install the required packages: 'qwen-tts', 'soundfile', and 'numpy' using pip.



In [ ]:
# ae923b6b
!pip install -U qwen-tts soundfile numpy

` # e0e12df2`  
**Reasoning**:
The required packages have been installed successfully. Now, I will re-run the main script to proceed with the voice cloning demonstration, which should resolve the previous `ModuleNotFoundError`.



` # 3a947465`  
**Reasoning**:
The previous code execution successfully ran the voice cloning demonstration, including model loading, voice profile creation, and generating audio files. All demos completed without errors, and the output confirmed the successful creation and saving of audio files. Therefore, the task is now complete.



In [ ]:
# ecNLeYx0BVNk
SYNTHESIS_LANGUAGE = "English"

In [ ]:
# 00c94542
import os

SCRIPT_FILE = 'script.txt'

# Create a dummy script.txt for demonstration purposes if it doesn't exist
if not os.path.exists(SCRIPT_FILE):
    with open(SCRIPT_FILE, 'w') as f:
        f.write("This is the first sentence from the script. This is the second sentence.\n\n")
        f.write("Here is another paragraph from the script file. This will be spoken by the cloned voice.")

# Read the content of script.txt
with open(SCRIPT_FILE, 'r') as f:
    script_content = f.read()

# Split the content into a list of strings, assuming paragraphs are separated by double newlines
script_content_list = [para.strip() for para in script_content.split('\n\n') if para.strip()]

print(f"Loaded {len(script_content_list)} paragraphs from {SCRIPT_FILE}")
display(script_content_list)


In [ ]:
# cellID PveTFwNmBWIM
# Your reference audio file (record yourself reading the REFERENCE_TEXT below)
REFERENCE_AUDIO = "sample.wav"

# The exact text you spoke in the reference audio
# IMPORTANT: This must match EXACTLY what you said in the recording

# Read REFERENCE_TEXT from sample.txt for utility program setup
SAMPLE_TEXT_FILE = 'sample.txt'

if not os.path.exists(SAMPLE_TEXT_FILE):
    # Create a dummy sample.txt for demonstration purposes if it doesn't exist
    # This content should ideally match your sample.wav
    with open(SAMPLE_TEXT_FILE, 'w') as f:
        f.write("This is a sample text for voice cloning demonstration. It should be spoken clearly and naturally.")

with open(SAMPLE_TEXT_FILE, 'r') as f:
    REFERENCE_TEXT = f.read().strip()

print(f"Loaded REFERENCE_TEXT from {SAMPLE_TEXT_FILE}:")
display(REFERENCE_TEXT)

` # f0180244`  
### Controlling Pitch and Tempo with the `instruct` parameter

While Qwen3-TTS doesn't offer explicit sliders for pitch and tempo, you can guide the speech generation to achieve these effects using the `instruct` parameter. This parameter takes a descriptive string that influences the tone, pace, and emotional quality of the synthesized speech.

**How to use it:**

You will find an `INSTRUCT` variable defined in one of the cells (cell `zvn6cFrkCoJ8`). You can modify this string to suggest desired speech characteristics.

**Examples:**

*   **Faster Tempo:**
    ```python
    INSTRUCT = "Speak quickly and clearly."
    ```
*   **Slower Tempo:**
    ```python
    INSTRUCT = "Speak slowly and deliberately, with pauses."
    ```
*   **Higher Pitch:**
    ```python
    INSTRUCT = "Maintain a higher, energetic pitch."
    ```
*   **Lower Pitch:**
    ```python
    INSTRUCT = "Speak in a deep, resonant, lower-pitched voice."
    ```
*   **Combined Effects:**
    ```python
    INSTRUCT = "Speak with a fast tempo, in an excited and high-pitched tone."
    ```
*   **Default/Neutral (or as currently set):**
    ```python
    INSTRUCT = "Warm, engaging voice with natural enthusiasm and varied intonation."
    ```

**Important Notes:**

*   The model interprets these instructions, so the effect might vary. Experiment with different phrases to find what works best for your desired outcome.
*   The effectiveness of the `instruct` parameter can depend on the model's training data and its ability to generalize from textual descriptions to acoustic features.
*   After changing `INSTRUCT`, you need to re-run the cells that use the `VoiceBot` or `VoiceCloner.speak` methods to apply the new instruction.

In [ ]:
# zvn6cFrkCoJ8
INSTRUCT="Warm, engaging voice with natural enthusiasm and varied intonation."

In [ ]:
# 1ea3e3f7
import os
import torch
import soundfile as sf
from pathlib import Path

# =============================================================================
# CONFIGURATION
# =============================================================================

# Output directory for generated audio
OUTPUT_DIR = Path("./generated_audio")


# =============================================================================
# DEVICE SETUP FOR APPLE SILICON (M4)
# =============================================================================

def get_device():
    """Determine the best available device for Apple Silicon."""
    if torch.backends.mps.is_available():
        print("✓ Using Apple Metal Performance Shaders (MPS)")
        return "mps"
    elif torch.cuda.is_available():
        print("✓ Using CUDA GPU")
        return "cuda:0"
    else:
        print("⚠ Using CPU (slower)")
        return "cpu"


# =============================================================================
# VOICE CLONER CLASS
# =============================================================================

class VoiceCloner:
    """
    A voice cloning wrapper for Qwen3-TTS optimized for Apple Silicon.
    """

    def __init__(self, use_small_model: bool = False):
        """
        Initialize the voice cloner.

        Args:
            use_small_model: Use 0.6B model instead of 1.7B (faster, less VRAM)
        """
        from qwen_tts import Qwen3TTSModel

        model_name = (
            "Qwen/Qwen3-TTS-12Hz-0.6B-Base" if use_small_model
            else "Qwen/Qwen3-TTS-12Hz-1.7B-Base"
        )

        self.device = get_device()
        print(f"Loading model: {model_name}")
        print("This may take a few minutes on first run...")

        # For Apple Silicon, use float16 instead of bfloat16
        dtype = torch.float16 if self.device == "mps" else torch.bfloat16

        # Load model with appropriate settings for M4
        self.model = Qwen3TTSModel.from_pretrained(
            model_name,
            device_map=self.device if self.device != "mps" else None,
            dtype=dtype,
            # Note: flash_attention_2 not available on MPS, omit for Apple Silicon
        )

        # Move to MPS if needed
        if self.device == "mps":
            self.model = self.model.to(self.device)

        self.voice_prompt = None
        print("✓ Model loaded successfully!")

    def create_voice_profile(self, audio_path: str, transcript: str):
        """
        Create a reusable voice profile from reference audio.

        Args:
            audio_path: Path to your voice sample (WAV, MP3, etc.)
            transcript: Exact text spoken in the audio
        """
        print(f"Creating voice profile from: {audio_path}")

        if not os.path.exists(audio_path):
            raise FileNotFoundError(f"Reference audio not found: {audio_path}")

        self.voice_prompt = self.model.create_voice_clone_prompt(
            ref_audio=audio_path,
            ref_text=transcript
        )
        print("✓ Voice profile created!")
        return self.voice_prompt

    def speak(self, text: str, language: str = "English",
          instruct: str = None, output_path: str = None):
        """
        Generate speech in your cloned voice.

        Args:
            text: Text to synthesize
            language: Target language
            output_path: Optional path to save the audio

        Returns:
            tuple: (audio_array, sample_rate)
        """
        if self.voice_prompt is None:
            raise ValueError("No voice profile loaded. Call create_voice_profile() first.")

        print(f"Generating: '{text[:50]}...' وصلت")

        wavs, sr = self.model.generate_voice_clone(
            text=text,
            language=language,
            voice_clone_prompt=self.voice_prompt,
            instruct=instruct
        )

        audio = wavs[0]

        if output_path:
            sf.write(output_path, audio, sr)
            print(f"✓ Saved to: {output_path}")

        return audio, sr

    def speak_direct(self, text: str, ref_audio: str, ref_text: str,
                     language: str = "English", output_path: str = None):
        """
        One-shot voice cloning without pre-creating a profile.

        Args:
            text: Text to synthesize
            ref_audio: Path to reference audio
            ref_text: Transcript of reference audio
            language: Target language
            output_path: Optional path to save audio

        Returns:
            tuple: (audio_array, sample_rate)
        """
        print(f"Cloning voice and generating: '{text[:50]}...' وصلت")

        wavs, sr = self.model.generate_voice_clone(
            text=text,
            language=language,
            ref_audio=ref_audio,
            ref_text=ref_text
        )

        audio = wavs[0]

        if output_path:
            sf.write(output_path, audio, sr)
            print(f"✓ Saved to: {output_path}")

        return audio, sr


# =============================================================================
# VOICE BOT CLASS (FOR CHATBOT INTEGRATION)
# =============================================================================

class VoiceBot:
    """
    A simple voice bot that speaks in your cloned voice.
    Ready for integration with LLMs like Claude, GPT, etc.
    """

    def __init__(self, ref_audio: str, ref_text: str, language: str = "English",
                 use_small_model: bool = False):
        """
        Initialize the voice bot with your voice.

        Args:
            ref_audio: Path to your voice sample
            ref_text: Transcript of your voice sample
            language: Default language for synthesis
            use_small_model: Use 0.6B model instead of 1.7B (faster, less VRAM)
        """
        self.cloner = VoiceCloner(use_small_model=use_small_model)
        self.cloner.create_voice_profile(ref_audio, ref_text)
        self.language = language
        self.output_dir = OUTPUT_DIR
        self.output_dir.mkdir(exist_ok=True)
        self.counter = 0

    def respond(self, text: str, instruct: str = None, save: bool = True):
        """
        Generate a spoken response in your voice.

        Args:
            text: The text to speak
            save: Whether to save the audio file

        Returns:
            tuple: (audio_array, sample_rate, file_path or None)
        """
        output_path = None
        if save:
            self.counter += 1
            output_path = str(self.output_dir / f"response_{self.counter:04d}.wav")

        audio, sr = self.cloner.speak(
            text=text,
            language=self.language,
            instruct=instruct,
            output_path=output_path
        )

        return audio, sr, output_path


# =============================================================================
# MAIN SCRIPT
# =============================================================================

def print_reference_texts():
    """Print the reference texts for recording."""
    print("\n" + "="*70)
    print("REFERENCE TEXTS FOR VOICE CLONING")
    print("="*70)
    print("\nRecord yourself reading ONE of these texts clearly:")
    print("\n--- ENGLISH (recommended for English bot) ---")
    print(REFERENCE_TEXT)
    print("\nTips for recording:")
    print("  • Use a quiet room")
    print("  • Speak naturally at your normal pace")
    print("  • Keep consistent distance from microphone")
    print("  • Save as WAV file (e.g., 'my_voice_sample.wav')")
    print("="*70 + "\n")


def main():
    """Main demonstration of voice cloning."""

    # Show reference texts for recording
    print_reference_texts()

    # Check if reference audio exists
    if not os.path.exists(REFERENCE_AUDIO):
        print(f"⚠ Reference audio not found: {REFERENCE_AUDIO}")
        print("\nTo get started:")
        print(f"  1. Record yourself reading the reference text above")
        print(f"  2. Save as '{REFERENCE_AUDIO}' in the same folder as this script")
        print(f"  3. Run this script again")
        return

    # Create output directory
    OUTPUT_DIR.mkdir(exist_ok=True)

    # ==========================================================================
    # DEMO 1: Simple voice cloning
    # ==========================================================================
    print("\n" + "="*70)
    print("DEMO 1: Basic Voice Cloning")
    print("="*70)

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print("✓ CUDA cache cleared and previous model deallocated.")

    cloner = VoiceCloner(use_small_model=True)
    cloner.create_voice_profile(REFERENCE_AUDIO, REFERENCE_TEXT)

    # ==========================================================================
    # DEMO 2: Voice Bot for chatbot integration
    # ==========================================================================
    print("\n" + "="*70)
    print("DEMO 2: Voice Bot (ready for LLM integration)")
    print("="*70)


    # Initialize bot with your voice
    bot = VoiceBot(
        ref_audio=REFERENCE_AUDIO,
        ref_text=REFERENCE_TEXT,
        language="English",
        use_small_model=True  # Set True for faster inference on limited RAM
    )

    for response in script_content_list: # Changed from bot_responses to script_content_list
        audio, sr, path = bot.respond(text=response, instruct=INSTRUCT)
        print(f"  Generated: {path}")

    # ==========================================================================
    # DONE
    # ==========================================================================
    print("\n" + "="*70)
    print("✓ All demos completed!")
    print(f"✓ Audio files saved to: {OUTPUT_DIR.absolute()}")
    print("="*70)
    print("\nNext steps:")
    print("  • Listen to the generated audio files")
    print("  • Integrate VoiceBot class with your LLM application")
    print("  • Adjust model size if needed (0.6B for faster, 1.7B for quality)")




# =============================================================================
# ENTRY POINT
# =============================================================================

if __name__ == "__main__":
  main()


In [ ]:
# 335953db
!pip install pydub

In [ ]:
# 5ab87797
import os
from pydub import AudioSegment
from pathlib import Path

# Define the directory where the generated audio files are located
audio_files_dir = Path('./generated_audio')

# Get all WAV files, ensuring they are sorted numerically
# This assumes a naming convention like 'response_0001.wav', 'response_0002.wav', etc.
wav_files = sorted(audio_files_dir.glob('response_*.wav'), key=lambda x: int(x.stem.split('_')[1]))

if not wav_files:
    print(f"No WAV files found in '{audio_files_dir}'. Please ensure audio generation was successful.")
else:
    print(f"Found {len(wav_files)} WAV files to combine.")

    # Initialize an empty AudioSegment
    combined_audio = AudioSegment.empty()

    # Concatenate each audio file
    for i, file_path in enumerate(wav_files):
        try:
            print(f"Combining file {i+1}/{len(wav_files)}: {file_path.name}")
            audio_segment = AudioSegment.from_wav(file_path)
            combined_audio += audio_segment
        except Exception as e:
            print(f"Error processing {file_path.name}: {e}")

    # Define the output path for the combined audio file
    output_filename = audio_files_dir / "combined_audio.wav"

    # Export the combined audio to a new WAV file
    try:
        combined_audio.export(output_filename, format="wav")
        print(f"\nSuccessfully combined all audio files into: {output_filename}")
        print("You can now download this file or play it directly below.")
    except Exception as e:
        print(f"Error exporting combined audio: {e}")


` # 713eec7d`  
### Play the Combined Audio

YouYou can use the following code to play the combined audio file directly within the notebook.

In [ ]:
# 575d7272
from IPython.display import Audio, display
from pathlib import Path

# output_filename = Path('./generated_audio/combined_audio.wav')
output_filename = Path('./combined_audio.wav')

if output_filename.exists():
    print(f"Playing: {output_filename.name}")
    display(Audio(output_filename))
else:
    print(f"Combined audio file '{output_filename.name}' not found.")


` # d3f802a9`  
## User Voice Capture and Configuration Guide

### Subtask:
Provide instructions on how to record your own voice, save it as 'my_voice_sample.wav', and update the 'REFERENCE_TEXT' variable in the script with the exact transcription. This step will enable you to clone your actual voice.

#### Instructions
1. Record yourself speaking the provided reference text clearly and naturally.
2. Save this recording as an audio file named `my_voice_sample.wav` in the same directory as the script.
3. Open the script and modify the `REFERENCE_TEXT` variable to exactly match the text you spoke in your `my_voice_sample.wav` recording.
4. Optionally, ensure the `SYNTHESIS_LANGUAGE` variable is set to 'English' for consistency, although the model handles multilingual output.

` # 02469d7e`  
## Final Task

### Subtask:
Summarize the voice cloning process, including how to use your own voice and explore the interactive mode. Provide guidance on next steps, such as integrating the VoiceBot or adjusting model parameters.


` # cellID bd9bb769`  
## Summary:

### Q&A
The voice cloning process involves installing necessary packages like `qwen-tts`, `soundfile`, and `numpy`. Once installed, a voice profile is created from an audio sample (`my_voice_sample.wav`), and then speech is generated in various languages (e.g., German, English) using the cloned voice.

To use your own voice:
1.  Record yourself speaking one of the provided reference texts (German or English) clearly.
2.  Save the recording as `my_voice_sample.wav` in the script's directory.
3.  Update the `REFERENCE_TEXT` variable in the script to exactly match the text you spoke in your `my_voice_sample.wav` recording. For example, if you recorded the German text, ensure `REFERENCE_TEXT` is set to `REFERENCE_TEXT_DE.strip()`.
4.  Optionally, set `SYNTHESIS_LANGUAGE` for consistency.

The interactive mode can be explored, and next steps include integrating the VoiceBot functionality and potentially adjusting model parameters for further customization.

### Data Analysis Key Findings
*   The required Python packages `qwen-tts` (version 0.0.5), `soundfile`, and `numpy` were successfully installed and/or upgraded, resolving initial `ModuleNotFoundError` issues.
*   The Qwen3-TTS voice cloning script executed successfully, utilizing a CUDA GPU for processing.
*   A voice profile was successfully created from a sample audio file (`my_voice_sample.wav`).
*   Speech was generated in both German and English using the cloned voice, demonstrating basic voice cloning and VoiceBot functionalities.
*   All generated audio files were saved to the `generated_audio` directory as expected.
*   Clear instructions were provided for users to record their own voice, save it as `my_voice_sample.wav`, and configure the `REFERENCE_TEXT` variable in the script to enable cloning of their actual voice.

### Insights or Next Steps
*   Users can now proceed to clone their own voice by following the provided instructions for recording and configuring the `my_voice_sample.wav` and `REFERENCE_TEXT` variables.
*   Explore the VoiceBot functionality and interactive mode as demonstrated in the script, and consider integrating the VoiceBot into other applications or adjusting model parameters for fine-tuning.
